In [1]:
import sys
import os

from pathlib import Path

from dotenv import load_dotenv
from pyspark.sql import SparkSession
import pandas as pd
import numpy as np

load_dotenv()

sys.path.append("../../")
from src.lib.mlflow import MlflowHandler
from src.main import _ensure_java_home
mlflow_handler = MlflowHandler()

In [2]:
spark_app_name = os.getenv("SPARK_APP_NAME")
spark_master_url = os.getenv("SPARK_MASTER_URL")
postgres_url = os.getenv("POSTGRES_URL")
postgres_user = os.getenv("POSTGRES_USER")
postgres_password = os.getenv("POSTGRES_PASSWORD")
_ensure_java_home()

required = {
        "POSTGRES_URL": postgres_url,
        "POSTGRES_USER": postgres_user,
        "POSTGRES_PASSWORD": postgres_password,
    }

spark = (
        SparkSession.builder.appName(spark_app_name)
        .master(spark_master_url)
        .config("spark.jars.packages", "org.postgresql:postgresql:42.7.8")
        .config("spark.ui.showConsoleProgress", "false")
        .config("spark.sql.adaptive.enabled", "true")
        .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
        .getOrCreate()
)

25/11/05 16:08:14 WARN Utils: Your hostname, MacBook-Air-de-Yose.local resolves to a loopback address: 127.0.0.1; using 10.48.79.23 instead (on interface en0)
25/11/05 16:08:14 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/Users/yosesotomayor/Code/store/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /Users/yosesotomayor/.ivy2/cache
The jars for the packages stored in: /Users/yosesotomayor/.ivy2/jars
org.postgresql#postgresql added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-0509887d-10af-4c14-85ed-9ead9874df37;1.0
	confs: [default]
	found org.postgresql#postgresql;42.7.8 in central
	found org.checkerframework#checker-qual;3.49.5 in central
:: resolution report :: resolve 152ms :: artifacts dl 9ms
	:: modules in use:
	org.checkerframework#checker-qual;3.49.5 from central in [default]
	org.postgresql#postgresql;42.7.8 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   2   |   0   |   0   |   0   ||   2   |   0   |
	---------------------------

In [3]:
missing = [k for k, v in required.items() if not v]
if missing:
        raise RuntimeError(
            f"Missing required environment variables for JDBC connection: {', '.join(missing)}"
        )

jdbc_options = {
        "url": str(postgres_url),
        "dbtable": "articles",
        "user": str(postgres_user),
        "password": str(postgres_password),
        "driver": "org.postgresql.Driver",
    }

reader = spark.read.format("jdbc")
for k, v in jdbc_options.items():
        reader = reader.option(k, v)

df = reader.load()

df.show()

+----------+------------+--------------------+---------------+-----------------+------------------+-----------------------+-------------------------+-----------------+-----------------+-------------------------+---------------------------+--------------------------+----------------------------+-------------+--------------------+----------+--------------------+--------------+----------------+----------+--------------------+----------------+------------------+--------------------+
|article_id|product_code|           prod_name|product_type_no|product_type_name|product_group_name|graphical_appearance_no|graphical_appearance_name|colour_group_code|colour_group_name|perceived_colour_value_id|perceived_colour_value_name|perceived_colour_master_id|perceived_colour_master_name|department_no|     department_name|index_code|          index_name|index_group_no|index_group_name|section_no|        section_name|garment_group_no|garment_group_name|         detail_desc|
+----------+------------+-------

In [4]:
jdbc_options_2 = {
        "url": str(postgres_url),
        "dbtable": "customers",
        "user": str(postgres_user),
        "password": str(postgres_password),
        "driver": "org.postgresql.Driver",
    }

reader_2 = spark.read.format("jdbc")
for k, v in jdbc_options_2.items():
        reader_2 = reader_2.option(k, v)

df_customers = reader_2.load()

df_customers.show()

+--------------------+----+------+------------------+----------------------+---+--------------------+----+-----+-------------+----------------+----------+
|         customer_id|  fn|active|club_member_status|fashion_news_frequency|age|         postal_code|name|email|password_hash|is_authenticated|created_at|
+--------------------+----+------+------------------+----------------------+---+--------------------+----+-----+-------------+----------------+----------+
|f55a003032a8c8f81...|NULL|  NULL|        PRE-CREATE|                  NONE| 36|86ab807568b3dd0a9...|NULL| NULL|         NULL|           false|      NULL|
|f55a0c2cc5df55162...|NULL|  NULL|            ACTIVE|                  NONE| 20|f084a0d336e36861b...|NULL| NULL|         NULL|           false|      NULL|
|f55a13eeb44c04f22...|NULL|  NULL|            ACTIVE|                  NONE| 27|a29ed3dd3b856183d...|NULL| NULL|         NULL|           false|      NULL|
|f55a245f40d0b8690...|NULL|  NULL|            ACTIVE|                 

In [5]:
df_customers_pandas = df_customers.toPandas()

In [6]:
df_pandas = df.toPandas()

In [7]:
pd.set_option('display.max_columns', None)
df_pandas

,article_id,product_code,prod_name,product_type_no,product_type_name,product_group_name,graphical_appearance_no,graphical_appearance_name,colour_group_code,colour_group_name,perceived_colour_value_id,perceived_colour_value_name,perceived_colour_master_id,perceived_colour_master_name,department_no,department_name,index_code,index_name,index_group_no,index_group_name,section_no,section_name,garment_group_no,garment_group_name,detail_desc
0,704524005,704524,Ester 3-Pack Body,256,Bodysuit,Garment Upper body,1010016,Solid,51,Light Pink,3,Light,4,Pink,6525,Baby Girl Jersey Fancy,G,Baby Sizes 50-98,4,Baby/Children,40,Baby Girl,1005,Jersey Fancy,Sleeveless bodysuits in soft cotton jersey wit...
1,704526001,704526,LOGG Gilmore long Parka.,262,Jacket,Garment Upper body,1010016,Solid,19,Greenish Khaki,4,Dark,20,Khaki green,1929,Outwear,A,Ladieswear,1,Ladieswear,2,H&M+,1007,Outdoor,Knee-length padded parka in woven fabric. Zip ...
2,704533001,704533,Cake Dress Set,270,Garment Set,Garment Full body,1010007,Embroidery,10,White,3,Light,9,White,6525,Baby Girl Jersey Fancy,G,Baby Sizes 50-98,4,Baby/Children,40,Baby Girl,1005,Jersey Fancy,"Set with a dress, puff pants and hairband in s..."
3,704536001,704536,Fennel Romper,267,Jumpsuit/Playsuit,Garment Full body,1010001,All over pattern,51,Light Pink,3,Light,4,Pink,6525,Baby Girl Jersey Fancy,G,Baby Sizes 50-98,4,Baby/Children,40,Baby Girl,1005,Jersey Fancy,Playsuit in soft cotton jersey with a round ne...
4,704536003,704536,Fennel Romper,267,Jumpsuit/Playsuit,Garment Full body,1010016,Solid,51,Light Pink,1,Dusty Light,4,Pink,6525,Baby Girl Jersey Fancy,G,Baby Sizes 50-98,4,Baby/Children,40,Baby Girl,1005,Jersey Fancy,Playsuit in soft cotton jersey with a round ne...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
105537,704522001,704522,Alexander cardigan,245,Cardigan,Garment Upper body,1010010,Melange,7,Grey,1,Dusty Light,12,Grey,8758,Young Boy Knitwear,I,Children Sizes 134-170,4,Baby/Children,47,Young Boy,1003,Knitwear,"Cardigan in soft, fine-knit cotton with a V-ne..."
105538,704524001,704524,Ester 3-Pack Body,256,Bodysuit,Garment Upper body,1010001,All over pattern,10,White,3,Light,9,White,6525,Baby Girl Jersey Fancy,G,Baby Sizes 50-98,4,Baby/Children,40,Baby Girl,1005,Jersey Fancy,Sleeveless bodysuits in soft cotton jersey wit...
105539,704524002,704524,Ester 3-Pack Body,256,Bodysuit,Garment Upper body,1010001,All over pattern,73,Dark Blue,4,Dark,2,Blue,6525,Baby Girl Jersey Fancy,G,Baby Sizes 50-98,4,Baby/Children,40,Baby Girl,1005,Jersey Fancy,Sleeveless bodysuits in soft cotton jersey wit...
105540,704524003,704524,Ester 3-Pack Body,256,Bodysuit,Garment Upper body,1010001,All over pattern,52,Pink,7,Medium,4,Pink,6525,Baby Girl Jersey Fancy,G,Baby Sizes 50-98,4,Baby/Children,40,Baby Girl,1005,Jersey Fancy,Sleeveless bodysuits in soft cotton jersey wit...


In [8]:
df_customers_pandas

,customer_id,fn,active,club_member_status,fashion_news_frequency,age,postal_code,name,email,password_hash,is_authenticated,created_at
0,f55a003032a8c8f810a465f64bf9e36bafd896d29524bc...,NaN,NaN,PRE-CREATE,NONE,36.0,86ab807568b3dd0a9cea497ac473d7910a839b411e774d...,None,None,None,False,NaT
1,f55a0c2cc5df55162161cce2f134980b3d0bdf9af5982d...,NaN,NaN,ACTIVE,NONE,20.0,f084a0d336e36861b4b1b7a7d4ad39df697b79683c01f4...,None,None,None,False,NaT
2,f55a13eeb44c04f22783670cadab66c56d38750a4beffc...,NaN,NaN,ACTIVE,NONE,27.0,a29ed3dd3b856183db3a6dc8fc6a92a2bc1b1a627874d0...,None,None,None,False,NaT
3,f55a245f40d0b86905b4f615ca25e1c34f1bc5d9fabc39...,NaN,NaN,ACTIVE,NONE,53.0,5ba37477d8e5c3af86682c81f0018ac6bb85966a2fc757...,None,None,None,False,NaT
4,f55a2a5115aae273613a9a730ec26f141d79d9272409a5...,NaN,NaN,ACTIVE,NONE,25.0,08ee43710825ec81cc002e340689feb4f7ee320615a4b0...,None,None,None,False,NaT
...,...,...,...,...,...,...,...,...,...,...,...,...
1371980,f559809eebef0671d5b5be737fabd134c913a29df98acb...,1.0,1.0,ACTIVE,Regularly,27.0,2c29ae653a9282cce4151bd87643c907644e09541abc28...,None,None,None,False,NaT
1371981,f5598e6e593f03740cbef42776cbf2dc20f379e8bebdb1...,1.0,1.0,ACTIVE,Regularly,24.0,2c29ae653a9282cce4151bd87643c907644e09541abc28...,None,None,None,False,NaT
1371982,f559ba6cd345bbe6f44547296bc03c906024441a6e5b33...,NaN,NaN,PRE-CREATE,NONE,NaN,502a058326ac4e090a3d3e1ffb53388f7c3713ad99bbe6...,None,None,None,False,NaT
1371983,f559beab8ebb0ec81ae812f6519cde4274250cf121f20e...,1.0,1.0,ACTIVE,Regularly,49.0,e0f349c59150ee8ab30f647c9b3ea38a1ef04f9494e25c...,None,None,None,False,NaT


In [9]:
df_transactions_pandas = pd.read_csv("/Users/yosesotomayor/Desktop/data_store/transactions_train.csv")

In [10]:
df_pandas['product_type_name'].nunique()

131

In [11]:
df_transactions_pandas = df_transactions_pandas[['customer_id', 'article_id']]

In [12]:
df_transactions_pandas.set_index(['customer_id'], inplace=True)

In [13]:
inter_ = df_transactions_pandas.index.intersection(df_customers_pandas['customer_id'])

In [23]:
df_transactions_pandas.groupby('customer_id').count()

,article_id
customer_id,
00000dbacae5abe5e23885899a1fa44253a17956c6d1c3d25f88aa139fdfc657,21
0000423b00ade91418cceaf3b26c6af3dd342b51fd051eec9c12fb36984420fa,86
000058a12d5b43e67d225668fa1f8d618c13dc232df0cad8ffe7ad4a1091e318,18
00005ca1c9ed5f5146b52ac8639a40ca9d57aeff4d1bd2c5feb1ca5dff07c43e,2
00006413d8573cd20ed7128e53b7b13819fe5cfc2d801fe7fc0f26dd8d65a85a,13
...,...
ffffbbf78b6eaac697a8a5dfbfd2bfa8113ee5b403e4747568cac33e8c541831,51
ffffcd5046a6143d29a04fb8c424ce494a76e5cdf4fab53481233731b5c4f8b7,84
ffffcf35913a0bee60e8741cb2b4e78b8a98ee5ff2e6a1778d0116cffd259264,45


In [26]:
prueba = df_transactions_pandas.loc["00000dbacae5abe5e23885899a1fa44253a17956c6d1c3d25f88aa139fdfc657"]

In [29]:
df_pandas.index = df_pandas.pop('article_id')
df_pandas.loc[prueba['article_id'].values]

,product_code,prod_name,product_type_no,product_type_name,product_group_name,graphical_appearance_no,graphical_appearance_name,colour_group_code,colour_group_name,perceived_colour_value_id,perceived_colour_value_name,perceived_colour_master_id,perceived_colour_master_name,department_no,department_name,index_code,index_name,index_group_no,index_group_name,section_no,section_name,garment_group_no,garment_group_name,detail_desc
article_id,,,,,,,,,,,,,,,,,,,,,,,,
625548001,625548,BB Chris puff jkt TP,262,Jacket,Garment Upper body,1010016,Solid,73,Dark Blue,4,Dark,2,Blue,8852,Young Boy Outdoor,I,Children Sizes 134-170,4,Baby/Children,45,Kids Outerwear,1007,Outdoor,"Padded jacket with a detachable hood, stand-up..."
176209023,176209,Mr Harrington w/hood,308,Hoodie,Garment Upper body,1010016,Solid,9,Black,4,Dark,5,Black,5283,Jacket Street,F,Menswear,3,Menswear,31,Mens Outerwear,1007,Outdoor,"Short, padded jacket with a jersey-lined hood ..."
627759010,627759,FLORA parka,262,Jacket,Garment Upper body,1010016,Solid,73,Dark Blue,4,Dark,2,Blue,7812,Kids Girl Outdoor,H,Children Sizes 92-140,4,Baby/Children,45,Kids Outerwear,1007,Outdoor,"Padded parka in woven fabric with a soft, brus..."
697138006,697138,Sophie jumpsuit,267,Jumpsuit/Playsuit,Garment Full body,1010001,All over pattern,51,Light Pink,1,Dusty Light,4,Pink,7616,Kids Girl Jersey Fancy,H,Children Sizes 92-140,4,Baby/Children,76,Kids Girl,1005,Jersey Fancy,Playsuit in cotton jersey with butterfly sleev...
568601006,568601,Mariette Blazer,264,Blazer,Garment Upper body,1010016,Solid,9,Black,4,Dark,5,Black,1212,Suit,A,Ladieswear,1,Ladieswear,11,Womens Tailoring,1008,Dressed,Fitted jacket in woven fabric with notch lapel...
568601006,568601,Mariette Blazer,264,Blazer,Garment Upper body,1010016,Solid,9,Black,4,Dark,5,Black,1212,Suit,A,Ladieswear,1,Ladieswear,11,Womens Tailoring,1008,Dressed,Fitted jacket in woven fabric with notch lapel...
607642008,607642,The Firm (1),259,Shirt,Garment Upper body,1010017,Stripe,9,Black,4,Dark,5,Black,1515,Blouse,A,Ladieswear,1,Ladieswear,11,Womens Tailoring,1010,Blouses,Top in a crêpe weave with a V-shaped opening a...
745232001,745232,Skirt Mini Stretch Edie,275,Skirt,Garment Lower body,1010023,Denim,9,Black,4,Dark,5,Black,1773,Denim Other Garments,D,Divided,2,Divided,57,Ladies Denim,1016,Trousers Denim,"Short 5-pocket skirt in washed, stretch denim ..."
656719005,656719,Serpente HW slim trouser,272,Trousers,Garment Lower body,1010016,Solid,9,Black,4,Dark,5,Black,1722,Trouser,A,Ladieswear,1,Ladieswear,15,Womens Everyday Collection,1009,Trousers,Tailored trousers in a stretch weave with two ...


In [48]:
df_pandas.info()

<class 'pandas.core.frame.DataFrame'>
Index: 105542 entries, 704524005 to 704524004
Data columns (total 24 columns):
 #   Column                        Non-Null Count   Dtype 
---  ------                        --------------   ----- 
 0   product_code                  105542 non-null  int32 
 1   prod_name                     105542 non-null  object
 2   product_type_no               105542 non-null  int32 
 3   product_type_name             105542 non-null  object
 4   product_group_name            105542 non-null  object
 5   graphical_appearance_no       105542 non-null  int32 
 6   graphical_appearance_name     105542 non-null  object
 7   colour_group_code             105542 non-null  int32 
 8   colour_group_name             105542 non-null  object
 9   perceived_colour_value_id     105542 non-null  int32 
 10  perceived_colour_value_name   105542 non-null  object
 11  perceived_colour_master_id    105542 non-null  int32 
 12  perceived_colour_master_name  105542 non-null  objec

In [51]:
df_transactions_pandas.info()

<class 'pandas.core.frame.DataFrame'>
Index: 31788324 entries, 000058a12d5b43e67d225668fa1f8d618c13dc232df0cad8ffe7ad4a1091e318 to fffef3b6b73545df065b521e19f64bf6fe93bfd450ab20e02ce5d1e58a8f700b
Data columns (total 1 columns):
 #   Column      Dtype
---  ------      -----
 0   article_id  int64
dtypes: int64(1)
memory usage: 517.3+ MB


In [61]:
from implicit.als import AlternatingLeastSquares
from scipy.sparse import coo_matrix

customer_map = {cid: i for i, cid in enumerate(df_transactions_pandas.index.unique())}
article_map = {aid: i for i, aid in enumerate(df_pandas.index)}

rows = df_transactions_pandas.index.map(customer_map.get)
cols = df_transactions_pandas['article_id'].map(article_map.get)
values = [1] * len(df_transactions_pandas)

interaction_matrix = coo_matrix((values, (rows, cols)))
interaction_matrix_csr = interaction_matrix.tocsr()

model = AlternatingLeastSquares(factors=50, regularization=0.1, iterations=15)
model.fit(interaction_matrix)

/Users/yosesotomayor/Code/store/.venv/lib/python3.11/site-packages/implicit/utils.py:164: ParameterWarning: Method expects CSR input, and was passed coo_matrix instead. Converting to CSR took 0.942986011505127 seconds
  warnings.warn(
100%|██████████| 15/15 [01:38<00:00,  6.55s/it]


In [91]:
inv_article_map = {v: k for k, v in article_map.items()}

random_customer_id = df_transactions_pandas.sample(1).index[0]
customer_idx = customer_map[random_customer_id]

recommended = model.recommend(
    customer_idx,
    interaction_matrix_csr[customer_idx],
    N=10,
    recalculate_user=True,              
    filter_already_liked_items=True     
)

if isinstance(recommended, list):
    item_indices = [it for it, _ in recommended]
    scores = [sc for _, sc in recommended]
elif isinstance(recommended, tuple) and len(recommended) == 2:
    item_indices, scores = recommended
else:
    raise TypeError(f"Formato inesperado de recommended: {type(recommended)}")

recommended_article_ids = [inv_article_map[int(i)] for i in item_indices]

out = (
    pd.DataFrame({
        "article_id": recommended_article_ids,
        "score": scores
    })
    .merge(
        df_pandas[["prod_name","product_type_name","garment_group_name","colour_group_name"]],
        on="article_id",
        how="left"
    )
    .sort_values("score", ascending=False, kind="mergesort")
    .reset_index(drop=True)
)

print(f"Cliente {random_customer_id} → historial-de-compras")
display(df_pandas.loc[df_transactions_pandas.loc[random_customer_id]['article_id'].values, ["prod_name","product_type_name","garment_group_name","colour_group_name"]])

print(f"Cliente {random_customer_id} → top-{len(out)} recomendaciones")
display(out.drop(columns=['score']))

Cliente a7ad8631996e456b484358381c945b5ef491694de55b8a1391155c769e30f2b1 → historial-de-compras


,prod_name,product_type_name,garment_group_name,colour_group_name
article_id,,,,
640807001,Lee long parka,Hoodie,Outdoor,Greenish Khaki
640807001,Lee long parka,Hoodie,Outdoor,Greenish Khaki
594834018,Dolly hood,Sweater,Jersey Fancy,Light Pink
660599014,Baraboom (1),Cardigan,Knitwear,Dark Beige
803969008,Inga bikini shopbasket 7pk,Underwear bottom,"Under-, Nightwear",Pink
372860002,7p Basic Shaftless,Socks,Socks and Tights,White


Cliente a7ad8631996e456b484358381c945b5ef491694de55b8a1391155c769e30f2b1 → top-10 recomendaciones


,article_id,prod_name,product_type_name,garment_group_name,colour_group_name
0,372860024,Basic 7p Shaftless,Socks,Socks and Tights,Light Grey
1,562245050,Luna skinny RW,Trousers,Trousers,White
2,575347003,7pk basic R sneaker socks,Socks,Socks and Tights,Black
3,736870001,Strap Top 2 pack,Vest top,Jersey Basic,Black
4,803757001,Bradley trousers,Trousers,Jersey Basic,Black
5,554598001,Nora T-shirt,T-shirt,Jersey Basic,Black
6,653188002,Sigge Sneaker Sock 5-p,Socks,"Under-, Nightwear",White
7,399256005,Skinny Ankle R.W Brooklyn,Trousers,Trousers Denim,White
8,575347014,7pk basic R sneaker socks,Socks,Socks and Tights,White
9,706016015,Jade HW Skinny Denim TRS,Trousers,Trousers,Dark Grey


In [ ]:
def dar_recomendaciones(customer_id):
    customer_idx = customer_map.get(customer_id)
    if customer_idx is None:
        raise ValueError(f"Cliente ID {customer_id} no encontrado en el mapa de clientes.")
    
    recommended = model.recommend(
        customer_idx,
        interaction_matrix_csr[customer_idx],
        N=10,
        recalculate_user=True,              
        filter_already_liked_items=True     
    )

    if isinstance(recommended, list):
        item_indices = [it for it, _ in recommended]
        scores = [sc for _, sc in recommended]
    elif isinstance(recommended, tuple) and len(recommended) == 2:
        item_indices, scores = recommended
    else:
        raise TypeError(f"Formato inesperado de recommended: {type(recommended)}")

    recommended_article_ids = [inv_article_map[int(i)] for i in item_indices]

    out = (
        pd.DataFrame({
            "article_id": recommended_article_ids,
            "score": scores
        })
        .merge(
            df_pandas[["prod_name","product_type_name","garment_group_name","colour_group_name"]],
            on="article_id",
            how="left"
        )
        .sort_values("score", ascending=False, kind="mergesort")
        .reset_index(drop=True)
    )
    return out

customer_id_ejemplo = df_transactions_pandas.sample(1).index[0]
recomendaciones = dar_recomendaciones(customer_id_ejemplo)
print(f"Recomendaciones para el cliente {customer_id_ejemplo}:")
display(recomendaciones.drop(columns=['score']))

Recomendaciones para el cliente b4bc187bbfd50a685ff3168c74aec453a20311ef8dc691dece40257a8fea3a0f:


,article_id,prod_name,product_type_name,garment_group_name,colour_group_name
0,741356002,Pamela Shorts HW,Shorts,Shorts,Blue
1,706016006,Jade HW Skinny Denim TRS,Trousers,Trousers,Dark Blue
2,399201005,Shaping Skinny H.W,Trousers,Trousers Denim,Dark Blue
3,612481002,ED Pamela (1),Shorts,Shorts,Blue
4,539723001,Jade Denim TRS,Trousers,Trousers,Black
5,579541001,Calista cardigan.,Cardigan,Knitwear,Black
6,624486001,Brit Baby Tee,T-shirt,Jersey Basic,Black
7,656763001,Kajsa HW,Shorts,Shorts,Black
8,824337001,Perrie Slim HW Denim Shorts,Shorts,Shorts,Blue
9,507909001,Rebecca or Delphine shirt,Shirt,Blouses,White
